In [ ]:
!pip install ollama

## Load gpt-oss:120b

In [2]:
from ollama import Client

ollama = Client(host="http://ollama.runai-shared.svc.cluster.local")

In [ ]:
MODEL = "gpt-oss:120b"

for progress in ollama.pull(MODEL, stream=True):
    print(progress)

# gpt-oss:120b :    size=65GB,  context=128K, thinking=True,  MMLU=90.00%, MMLU-Pro=NA,     GPQA-Diamond=80.1%
# llama3.3:70b :    size=43GB,  context=128K, thinking=False, MMLU=86.0%,  MMLU-Pro=68.9%,  GPQA-Diamond=50.5%
# deepseek-r1:70b : size=43GB,  context=128K, thinking=True,  MMLU=NA,     MMLU-Pro=NA,     GPQA-Diamond=71.5%
# deepseek-r1:8b :  size=5.2GB, context=128K, thinking=True,  MMLU=NA,     MMLU-Pro=NA,     GPQA-Diamond=81.0%
# cogito:70b :      size=43GB,  context=128K, thinking=True,  MMLU=91.00%, MMLU-Pro=78.47%, GPQA-Diamond=60.61%
# qwen3:30b :       size=19GB,  context=256K, thinking=True,  MMLU=NA,     MMLU-Pro=NA,     GPQA=70.4%

# MMLU: Measuring Massive Multitask Language Understanding (https://arxiv.org/abs/2009.03300)
# GPQA: A Graduate-Level Google-Proof Q&A Benchmark (https://arxiv.org/abs/2311.12022) 
# ARC: Abstraction and Reasoning Corpus (https://arxiv.org/abs/2409.01374) 

In [48]:
for mod in ollama.list().models:
    print(mod)
    print('\n')

model='gpt-oss:120b' modified_at=datetime.datetime(2026, 1, 20, 0, 17, 0, 538237, tzinfo=TzInfo(UTC)) digest='a951a23b46a1f6093dafee2ea481d634b4e31ac720a8a16f3f91e04f5a40ecd9' size=65369818941 details=ModelDetails(parent_model='', format='gguf', family='gptoss', families=['gptoss'], parameter_size='116.8B', quantization_level='MXFP4')




## Load Scopus data

In [ ]:
!pip install pyreadr

In [5]:
import numpy as np
import pandas as pd
import requests
import json
import time
import pyreadr
from openpyxl import Workbook, load_workbook
from pathlib import Path
import importlib
import functions
importlib.reload(functions)

SEED = 1

# Show all columns without truncation
pd.set_option("display.max_colwidth", None)  

# If you also want to see all rows/columns
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

In [30]:
# ----------------------------------------------------------------
# Load data

result = pyreadr.read_r("scopus/scopus_complete.rds")
scopus = result[None]   
scopus = scopus.drop_duplicates(subset="EID", keep="first").reset_index(drop=True)
print(len(scopus))
print(len(np.unique(scopus['EID'])))
print(min(scopus['Year']))
print(max(scopus['Year']))

112830
112830
1914.0
2025.0


In [13]:
eid = '2-s2.0-80053068062'
print(str(scopus[scopus['EID']==eid]['Title'].values[0]) + "\n" + 
      "Abstract: " + str(scopus[scopus['EID']==eid]['Abstract'].values[0]) + "\n" + 
      "Author keywords: " + str(scopus[scopus['EID']==eid]['Author Keywords'].values[0]) + "\n" + 
      "Index keywords: " + str(scopus[scopus['EID']==eid]['Index Keywords'].values[0]))

A survey on pest insect fauna of safflower fields in the Iranian Province of Kohgiloyeh and Boyerahmad
Abstract: Safflower (Carthamus tinctorius L.) is an important oilseed crop and an essential component of cropping systems in the dry regions and marginal areas of the world. Since the identification of insect pests and their natural enemies is one of the most efficient methods for pest management, an attempt was made to investigate insect fauna in safflower fields of Kohgiloyeh and Boyerahmad Provinces, Iran. To accomplish this, three main cultivation regions were selected and insect pests and their natural enemies were surveyed and collected according to the plant phenology during 2008 to 2009. To sample different techniques, such as sweeping, sticky yellow traps, bait traps and rearing of adult insects, were used. Pest insect fauna was as follows: 1) Pest insects feeding inside the flower heads (bolls) and their natural enemies: Acanthiophilus helianthi, Chaetorellia carthami, Trell

## Load prompts 

In [11]:
prompt_format_species = {
    #"title": "SpeciesList",
    "type": "object",
    "properties": {
        # "all_species": {
        #     "type": "array", 
        #     "items": {"type": "string"},
        #     "description":"List of all species mentioned in the text. Only list species that are mentioned at the genus or species level (latin binomials)."
        # },
        "herbivores": {
            "type": "array", 
            "items": {"type": "string"},
            "description":"List of all herbivorous species mentioned in the text. Only list herbivorous species that are mentioned at the genus or species level (latin binomials) and only list species from the main abstract - don't list species that are only mentioned in the keywords."
        },
        "natural_enemies": {
            "type": "array",
            "items": {"type": "string"},
            "description":"List of natural enemies (e.g., predators and parasitoids) mentioned in the text. Only list natural enemies that are mentioned at the genus or species level (latin binomials) and only list species from the main abstract - don't list species that are only mentioned in the keywords. Ignore species that are solely mentioned as hyperparasitoids or predators of other natural enemies."
        }
    },
    "required": ["herbivores", "natural_enemies"],
    "additionalProperties": False
}

In [12]:
prompt_format_herbivore = {
    #"title": "HerbivoreTable",
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "taxonomy": {
                "type": "object",
                "properties": {
                    "phylum": {
                        "type": "string",
                        "description": "Taxonomic phylum of the species",
                        "default": ""
                    },
                    "class": {
                        "type": "string",
                        "description": "Taxonomic class of the species",
                        "default": ""
                    },
                    "order": {
                        "type": "string",
                        "description": "Taxonomic order of the species",
                        "default": ""
                    },
                    "family": {
                        "type": "string",
                        "description": "Taxonomic family of the species",
                        "default": ""
                    },
                    "genus": {
                        "type": "string",
                        "description": "Taxonomic genus of the species",
                        "default": ""
                    },
                    "species": {
                        "type": "string",
                        "description": "Taxonomic species name (latin binomial) or 'spp.'",
                        "default": ""
                    }
                },
                "required": ["phylum", "class", "order", "family", "genus", "species"],
                "additionalProperties": False
            },
            "common_name": {
                "type": "string",
                "description": (
                    "Only if the species has a common name associated with it in the text, state it here. Example: The common name of the sugarcane froghopper (Monecphora saccharina) is 'The sugarcane froghopper'."
                               ),
                "default": ""
            },
            "feeding_mode": {
                "type": "string", 
                "enum": ["granivory", "frugivory","gall formation", "herbivory"],
                "description": (
                    "Select if the species is a herbivore through granivory (seed predation), frugivory (fruit consumption), gall formation, or simply herbivory (default; any other form of herbivory, or unspecified herbivory)."
                               ),
                "default": "herbivory"
            },
            "host_plants": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "host_plant": {
                            "type":"string",
                            "description":(
                                "Plant or crop mentioned as host of the herbivorous species or as affected by it (add latin binomial if possible). This needs to be the plant specifically stated in the text to be affected or used as host, and should not be inferred from the herbivore's common name (e.g., if a herbivore called the 'leek moth' is solely stated to be a pest of onions, then the host plant is onion, not leek). "
                                "If a herbivorous species is stated to occur on a particular plant, or within a particular plantation/crop field, then it may be inferred to be a herbivore of this plant/crop. "
                                "If the herbivore is stated or implicated as a general pest of agriculture, forestry or ornamentals without any particular host plants, then simply state 'agricultural crops', 'forestry' or 'ornamentals', respectively. "
                                "Only select plants that are stated in the main abstract text - ignore plants that are only mentioned in the keywords."
                            )
                        },
                        "herbivore_is_pest_of_plant": {
                            "type": "boolean",
                            "description": ("Is the herbivore a pest of this plant (True or False)? "
                                            "The species must either be mentioned explicitly as a pest of this plant, or at least as being the target of biological control (e.g., target of biocontrol study, pesticide application, etc.), or being the cause of destruction, infestation, commercial damage, etc., in relation to this plant. "
                                            "Pay close attention to the host crop or plant: If this is a weed or invasive, then the herbivore cannot be a pest of this plant; it can only be a pest if it affects plants that have commercial purposes in agriculture or forestry."
                                           ),
                            "default": False
                        },
                        "pest_importance": {
                            "type":"string",
                            "enum":["minor","major","NA"],
                            "description": (
                                "If the herbivore is a pest of this plant, is it of minor or major importance? "
                                "Minor: The pest is stated to be of little economic importance or is only a nuisance and does not require biocontrol measures. "
                                "Major: Major: The herbivore is mentioned to be an important pest (e.g., highly destructive, commercially relevant, dominant, widespread pest, etc.), or is the intended target of biological control (e.g., target of a biocontrol or pesticide study) with regards to this plant/crop. "
                                "Select 'NA' if nothing to this effect is mentioned or suggested in the text."
                            ),
                            "default":"NA"
                        },
                        "biological_control_agent": {
                            "type":"boolean",
                            "description": "If the plant is a weed or is invasive, is the herbivore stated to be a biological control agent of the plant, or a promising/likely candidate for biological control (True or False)?",
                            "default": False
                        }
                    },
                    "required": ["host_plant", "herbivore_is_pest_of_plant"],
                    "additionalProperties": False
                }
            },
            "associated_industries": {
                  "type": "array",
                  "items": {
                    "type": "string",
                    "enum": [
                      "arable crops",
                      "greenhouse production",
                      "pasture",
                      "fruit plantations",
                      "forestry and timber production",
                      "stored grains",
                      "ornamental and horticultural plants", 
                      "unspecified agriculture",
                      "other",
                      "NA"
                    ]
                  },
                  "uniqueItems": True,
                  "description": (
                      "List of any industries that are related to the host plants or crops of this herbivore, or are implied/mentioned in the text. "
                      "If none of these industries are mentioned explicitly, choose whichever industries are most applicable to the host plants/crops of this herbivore. "
                      "Choose 'unspecified agriculture' if none apply and agriculture is only mentioned in a general sense and no specific host plants are mentioned, and choose 'other' if it's an industry other than agriculture or forestry. " 
                      "Choose NA if no information about this is provided in the text whatsoever."
                  ),
                  "default": []
            },
            "herbivore_invasive_in": {
              "type":"array",
                "items": {"type": "string"},
              "description": "If this herbivore is explicitly described with the term 'invasive', in which countries or regions is it stated to be invasive?"
            },
            "pathogen_vector": {
                "type":"array",
                  "items": {"type": "string"},
                "description": "If the herbivore is stated to be a vector of a pathogen or disease, state the pathogen(s) and/or disease(s) here."
            },
            "synonyms": {
                "type":"array",
                "items": {"type": "string"},
              "description": "Any latin synonyms of this species stated in the text. These must be latin binomials, not common names."
            }
        },
        "required": ["taxonomy", "feeding_mode", "host_plants", "associated_industries"],
        "additionalProperties": False
    }
}

In [13]:
prompt_format_enemy = {
    #"title": "NaturalEnemyTable",
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "taxonomy": {
                "type": "object",
                "properties": {
                    "phylum": {
                        "type": "string",
                        "description": "Taxonomic phylum of the species",
                        "default": ""
                    },
                    "class": {
                        "type": "string",
                        "description": "Taxonomic class of the species",
                        "default": ""
                    },
                    "order": {
                        "type": "string",
                        "description": "Taxonomic order of the species",
                        "default": ""
                    },
                    "family": {
                        "type": "string",
                        "description": "Taxonomic family of the species",
                        "default": ""
                    },
                    "genus": {
                        "type": "string",
                        "description": "Taxonomic genus of the species",
                        "default": ""
                    },
                    "species": {
                        "type": "string",
                        "description": "Taxonomic species name (latin binomial) or 'spp.'",
                        "default": ""
                    }
                },
                "required": ["phylum", "class", "order", "family", "genus", "species"],
                "additionalProperties": False
            },
            "common_name": {
                "type": "string",
                "description": (
                    "Only if the species has a common name associated with it in the text, state it here. Example: The common name of the sugarcane froghopper (Monecphora saccharina) is 'The sugarcane froghopper'."
                               ),
                "default": ""
            },
            "feeding_mode": {
                "type": "string", 
                "enum": ["predation", "parasitism", "NA"],
                "description": (
                    "Select if the species is a natural enemy through predation or parasitism (which includes parasitoidism). This may be inferred from the taxonomy (e.g. families of parasitoid wasps or predators). If not mentioned in the text or unclear, select NA."
                               )
            },
            "prey_species": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "herbivorous_prey": {
                            "type": "string",
                            "description": (
                                "Name of the herbivorous species (binomial if possible) that is preyed upon or used as host by the natural enemy. These may also be groups of species (e.g. bark beetles, lepidopteran stemborers, etc., if only stated as such in the text). " 
                                "Look out for mentions of consumption: If a natural enemy is stated to consume a herbivorous species, this is evidence for the species being a natural enemy of the species."
                            )
                        },
                        "important_natural_enemy": {
                            "type":"boolean",
                            "description": (
                                "Is the natural enemy stated to be an important natural enemy of this herbivorous prey/host (True or False)? "
                                "Select True if the natural enemy is mentioned to be an important, dominant, frequent, common (etc.) natural enemy, is noted to exert important/effective/significant suppression/predation/parasitism, makes up a large proportion of the predation/parasitism of this herbivore, or has the potential to control/reduce/suppress the herbivore. "
                                "Note: If the predation/parasitism of the natural enemy is found to be comparable to (or not found to be significantly different from) another species that is held to be an important natural enemy, then both species are important natural enemies." 
                            ),
                            "default": False
                        },
                        "biological_control": {
                            "type": "boolean",
                            "description": (
                                "Does the text explicitly state that the natural enemy provides biological control of this herbivorous prey/host (True or False)? "
                                "Important: Select True ONLY if the natural enemy has been shown to provide/exert control of this herbivore or is successfully used as a biological control agent of this herbivore. "
                                "If the text simply states that the natural enemy is a biological control agent of the herbivore, then assume that this means that it is successful - however, if the text mentions that the species was subsequently found not to provide effective control, then biological_control should be False. "
                                "Important: If a natural enemy is mentioned to have 'potential' to control a herbivorous pest, then this makes it an important natural enemy of the herbivore (important_natural_enemy is True) - however, it does not suffice for biological_control to be True. "
                                "biological_control should only be True if the natural enemy is found to effectively control the herbivorous pest, or if it is used or has been used successfully as a biological control agent."
                            ),
                            "default": False
                        }
                    },
                    "required": ["herbivorous_prey"],
                    "additionalProperties": False
                },
                "description": (
                    "List of herbivorous species that the natural enemy is stated to prey upon, use as hosts, control, etc. (if stated in the text)."
                )
            },
            "associated_plants": {
                "type": "string",
                "description": (
                    "List of any plants associated with this natural enemy (e.g., where it is found to occur) or its herbivorous prey/hosts of (e.g., the affected crops, fruits, trees, flowers, etc.). "
                    #"These should be commerically relevant plants or crops, not weeds or wild/native flora."
                ),
                "default": ""
            },
            "associated_industries": {
              "type": "array",
              "items": {
                "type": "string",
                "enum": [
                  "arable crops",
                  "greenhouse production",
                  "pasture",
                  "fruit plantations",
                  "forestry and timber production",
                  "stored grains",
                  "ornamental and horticultural plants", 
                  "unspecified agriculture",
                  "other",
                  "NA"
                ]
              },
              "uniqueItems": True,
              "description": (
                  "List of any industries that are related to the host plants or crops associated with this natural enemy and/or its herbivorous prey/hosts. "
                  "If none of these industries are mentioned explicitly, choose whichever industries are most applicable to the plants/crops associated with this natural enemy or its prey/hosts. " 
                  "Choose 'unspecified agriculture' if none apply and agriculture is only mentioned in a general sense and no specific host plants are mentioned, and choose 'other' if it's an industry other than agriculture or forestry. "
                  "Choose NA if no information about this is provided in the text whatsoever."
              ),
              "default": []
            },
            "hyperparasitoids": {
              "type":"array",
                "items": {"type": "string"},
              "description": "If the natural enemy is a parasitoid and the text mentions any hyperparasitoids parasitising this natural enemy, then list these hyperparasitoids here."
            },
            "invasive_in": {
              "type":"array",
                "items": {"type": "string"},
              "description": "If this natural enemy is explicitly described with the term 'invasive', in which countries or regions is it stated to be invasive?"
            },
            "vectors": {
                "type":"array",
                  "items": {"type": "string"},
                "description": "If this natural enemy is a natural enemy of a herbivore due to vectoring a symbiotic bacterium/pathogen/disease that leads to mortality of the herbivorous prey/hosts, state it here.",
            },
            "synonyms": {
                "type":"array",
                "items": {"type": "string"},
              "description": "Any latin synonyms of this species stated in the text. These must be latin binomials, not common names."
            }
        },
        "required": [
            "taxonomy", "feeding_mode", "prey_species", "associated_industries"
        ],
        "additionalProperties": False
    },
    "description": "Provide an item for each natural enemy in the text."
}

In [14]:
cols = ["Phylum", "Class", "Order", "Family", "Genus", "Species", "Common Name", "Type", "Feeding Mode",
        "Host Plants", "Herbivore Is Pest", "Pest Importance","Herbivore Is BCA","Herbivore Prey", "Important Natural Enemy","Biological Control",
        "Associated Plants","Associated Industries","Hyperparasitoids", "Invasive","Vectors","Synonyms", "Generated Response", "Abstract", "EID"]

# system + user prompts

# system prompt for generating text: 
with open("system_prompt.txt") as f:
    system_prompt_text = f.read()
    
# system prompt for extracting information as JSON: 
with open("system_prompt-gpt-oss.txt") as f: 
    system_prompt_json = f.read() 

# user prompt for extracting herbivorous species and their natural enemies: 
with open("prompt_v5-simple.txt") as f:
    user_prompt_simple = f.read()

# user prompt for extracting information from herbivorous species: 
with open("prompt_v5-simple-pests.txt") as f:
    user_prompt_herbivores = f.read()

# user prompt for extracting information from natural enemies of herbivorous species: 
with open("prompt_v5-simple-enemies.txt") as f:
    user_prompt_enemies = f.read()

## Example step-by-step usage: 

In [79]:
eid = "2-s2.0-80053068062" #"2-s2.0-80053068062" #"2-s2.0-0000252648" # "2-s2.0-0001688927" #"2-s2.0-80053068062"#"2-s2.0-77957191872" #"2-s2.0-0000252648"#"2-s2.0-0027065837"#"2-s2.0-77957191872" #"2-s2.0-0000252648"#"2-s2.0-0001688927" #"2-s2.0-0036212873" #"2-s2.0-85019767155" #"2-s2.0-0028166139" #"2-s2.0-0021593437" #"2-s2.0-0000252648" #"2-s2.0-0024185618"#"2-s2.0-0031400560" 
row = scopus[scopus["EID"] == eid].iloc[0]
title = str(row.get("Title", ""))
abstract_text = str(row.get("Abstract", ""))
author_keys = row.get("Author Keywords", "")
index_keys = row.get("Index Keywords", "")

abstract = (
    f"{title}. Abstract: {abstract_text}"
    + (f"\nAuthor keywords: {author_keys}")#if pd.notna(author_keys) else "")
    + (f"\nIndex keywords: {index_keys}")# if pd.notna(index_keys) else "")
)

# extracting lists of herbivores and natural enemies: 
species_list, resp_text = functions.get_species_list(ollama, 
                                                     MODEL, 
                                                     SEED,
                                                     abstract, 
                                                     system_prompt_text, 
                                                     system_prompt_json,
                                                     user_prompt_simple, 
                                                     prompt_format_species)
print(species_list)

{'herbivores': ['Acanthiophilus helianthi', 'Chaetorellia carthami', 'Trellia luteola', 'Oxycarenus palens', 'Oxycarenus hyalipennis', 'Lygus spp.', 'Uroleucon compositae', 'Pleotrichophorus glandolosus', 'Brachycaudus helichrysi', 'Chromatomyia horticola', 'Agrotis sp.', 'Helicoverpa peltigera', 'Thrips tabaci', 'Euscelis alsius', 'Macrosteles laevis', 'Psammotettix striatus', 'Circulifer haematoceps', 'Reptalus bitinctus'], 'natural_enemies': ['Antistrophoplex conthurnatus', 'Microdontomenu annulatus', 'Bracon hebetor', 'Bracon luteator', 'Pronotalia carlinarum', 'Ormyrus orientalis', 'Colotrechnus viridis', 'Pteromalus sp.', 'Eurytoma acroptilae', 'Isocolus tinctorius', 'Lysiphlebus fabarum', 'Pachyneuron concolor', 'Coccinella septempunctata', 'Hippodamia variegata', 'Orius spp.', 'Chrysoperla carnea', 'Neoaliturus fenestratus', 'Aeolothrips collaris', 'Haplothrips sp.']}


In [81]:
len(species_list['herbivores']), len(species_list['natural_enemies'])

(18, 19)

In [82]:
species_list['herbivores']

['Acanthiophilus helianthi',
 'Chaetorellia carthami',
 'Trellia luteola',
 'Oxycarenus palens',
 'Oxycarenus hyalipennis',
 'Lygus spp.',
 'Uroleucon compositae',
 'Pleotrichophorus glandolosus',
 'Brachycaudus helichrysi',
 'Chromatomyia horticola',
 'Agrotis sp.',
 'Helicoverpa peltigera',
 'Thrips tabaci',
 'Euscelis alsius',
 'Macrosteles laevis',
 'Psammotettix striatus',
 'Circulifer haematoceps',
 'Reptalus bitinctus']

In [83]:
species_list['natural_enemies']

['Antistrophoplex conthurnatus',
 'Microdontomenu annulatus',
 'Bracon hebetor',
 'Bracon luteator',
 'Pronotalia carlinarum',
 'Ormyrus orientalis',
 'Colotrechnus viridis',
 'Pteromalus sp.',
 'Eurytoma acroptilae',
 'Isocolus tinctorius',
 'Lysiphlebus fabarum',
 'Pachyneuron concolor',
 'Coccinella septempunctata',
 'Hippodamia variegata',
 'Orius spp.',
 'Chrysoperla carnea',
 'Neoaliturus fenestratus',
 'Aeolothrips collaris',
 'Haplothrips sp.']

In [84]:
print(resp_text)

**Herbivorous (phytophagous) species mentioned in the abstract**  
*(only those given with a genus‑ or species‑level name; “spp.” is accepted as a genus‑level entry)*  

| # | Species (genus / species) | Reason it is considered herbivorous |
|---|---------------------------|--------------------------------------|
| 1 | *Acanthiophilus helianthi* | Tephritid fly whose larvae develop inside safflower flower heads (bolls). |
| 2 | *Chaetorellia carthami* | Tephritid fly that feeds/larvae develop inside safflower flower heads. |
| 3 | *Trellia luteola* | Tephritid fly (family Tephritidae) that attacks safflower flower heads. |
| 4 | *Oxycarenus palens* | True bug (Lygaeidae) recorded as a pest feeding around safflower bolls. |
| 5 | *Oxycarenus hyalipennis* | Same family as above; reported as a pest around safflower bolls. |
| 6 | *Lygus* spp. | Plant‑feeding bugs (Lygaeidae) listed as pests feeding around the flower heads. |
| 7 | *Uroleucon compositae* | Aphid that feeds on safflower fol

In [66]:
# extract information for natural enemies as text: 

user_prompt_initial=f"""
                     Here are a title, abstract and keywords: {abstract}.\n
                     Provide information for the following natural enemies: {', '.join(species_list["natural_enemies"])}.
                     {user_prompt_enemies}\n
                     Explain your reasoning.
                     """

chat_history = [
    {"role": "system", "content": system_prompt_text},
    {"role": "user", "content": user_prompt_initial} 
]

resp = functions.llm_generate(ollama,
                              MODEL, 
                              SEED, 
                              chat_history)

In [67]:
print(resp)

**Extracted information for the requested natural enemies**

| # | Natural enemy | Taxonomy (Phylum‑Class‑Order‑Family‑Genus‑Species) | Common name | Feeding mode* | Herbivorous prey species (as stated in the abstract) | Important natural enemy? | Biological‑control proven? | Associated plant(s) (from abstract) | Associated industry (most applicable) | Hyperparasitoids | Vectors | Synonyms |
|---|---------------|---------------------------------------------------|-------------|---------------|------------------------------------------------------|--------------------------|----------------------------|-------------------------------------|----------------------------------------|------------------|---------|----------|
| 1 | **Lysiphlebus fabarum** | Animalia – Arthropoda – Insecta – Hymenoptera – Braconidae – *Lysiphlebus* – *fabarum* | Aphid‑parasitoid wasp | Parasitism | • *Uroleucon compositae*  (aphid)  <br>• *Pleotrichophorus glandolosus* (aphid)  <br>• *Brachycaudus helichrysi* 

In [68]:
# extract this information as JSON: 

user_prompt=f"""
             Return the following natural enemies: {', '.join(sp_list)}
             Provide the information following this JSON schema exactly and output JSON only: {prompt_format_enemy}
             """

chat_history = [
        {"role": "system", "content": system_prompt_json},
        {"role": "user", "content": user_prompt_initial}] + [
        {"role": "assistant", "content": resp},
        {"role": "user", "content": user_prompt}]

resp_json = functions.llm_generate(ollama, 
                                   MODEL, 
                                   SEED, 
                                   chat_history)

species_json = string_to_json(resp_json)

In [69]:
species_json

[{'taxonomy': {'phylum': 'Animalia',
   'class': 'Insecta',
   'order': 'Hymenoptera',
   'family': 'Braconidae',
   'genus': 'Lysiphlebus',
   'species': 'fabarum'},
  'common_name': '',
  'feeding_mode': 'parasitism',
  'prey_species': [{'herbivorous_prey': 'Uroleucon compositae',
    'important_natural_enemy': False,
    'biological_control': False},
   {'herbivorous_prey': 'Pleotrichophorus glandolosus',
    'important_natural_enemy': False,
    'biological_control': False},
   {'herbivorous_prey': 'Brachycaudus helichrysi',
    'important_natural_enemy': False,
    'biological_control': False}],
  'associated_plants': 'Carthamus tinctorius',
  'associated_industries': ['arable crops'],
  'hyperparasitoids': [],
  'invasive_in': [],
  'vectors': [],
  'synonyms': []},
 {'taxonomy': {'phylum': 'Animalia',
   'class': 'Insecta',
   'order': 'Hymenoptera',
   'family': 'Pteromalidae',
   'genus': 'Pachyneuron',
   'species': 'concolor'},
  'common_name': '',
  'feeding_mode': 'parasit

In [71]:
df = get_model_responses(system_prompt, species_list, abstract, prompt_format_herbivore, prompt_format_enemy, cols)
df_ = df.map(flatten_cell)
df_.drop(columns=["Generated Response"])

,Phylum,Class,Order,Family,Genus,Species,Common Name,Type,Feeding Mode,Host Plants,Herbivore Is Pest,Pest Importance,Herbivore Is BCA,Herbivore Prey,Important Natural Enemy,Biological Control,Associated Plants,Associated Industries,Hyperparasitoids,Invasive,Vectors,Synonyms,Abstract,EID
0,Arthropoda,Insecta,Diptera,Tephritidae,Acanthiophilus,Acanthiophilus helianthi,,herbivore,herbivory,Carthamus tinctorius L.,True,NA,False,,,,,arable crops,,,,,,
1,Arthropoda,Insecta,Diptera,Tephritidae,Chaetorellia,Chaetorellia carthami,,herbivore,herbivory,Carthamus tinctorius L.,True,NA,False,,,,,arable crops,,,,,,
2,Arthropoda,Insecta,Diptera,Tephritidae,Trellia,Trellia luteola,,herbivore,herbivory,Carthamus tinctorius L.,True,NA,False,,,,,arable crops,,,,,,
3,Arthropoda,Insecta,Hemiptera,Lygaeidae,Oxycarenus,Oxycarenus palens,,herbivore,herbivory,Carthamus tinctorius L.,True,NA,False,,,,,arable crops,,,,,,
4,Arthropoda,Insecta,Hemiptera,Lygaeidae,Oxycarenus,Oxycarenus hyalipennis,,herbivore,herbivory,Carthamus tinctorius L.,True,NA,False,,,,,arable crops,,,,,,
5,Arthropoda,Insecta,Hemiptera,Miridae,Lygus,spp.,,herbivore,herbivory,Carthamus tinctorius,True,NA,False,,,,,arable crops,,,,,,
6,Arthropoda,Insecta,Hemiptera,Aphididae,Uroleucon,compositae,,herbivore,herbivory,Carthamus tinctorius,True,NA,False,,,,,arable crops,,,,,,
7,Arthropoda,Insecta,Hemiptera,Aphididae,Pleotrichophorus,glandolosus,,herbivore,herbivory,Carthamus tinctorius,True,NA,False,,,,,arable crops,,,,,,
8,Arthropoda,Insecta,Hemiptera,Aphididae,Brachycaudus,helichrysi,,herbivore,herbivory,Carthamus tinctorius,True,NA,False,,,,,arable crops,,,,,,
9,Arthropoda,Insecta,Lepidoptera,Noctuidae,Agrotis,sp.,,herbivore,herbivory,Carthamus tinctorius,True,NA,False,,,,,arable crops,,,,,,


In [72]:
print(df_["Generated Response"][0])

**Explanation of the extraction process**

The abstract reports a survey of insects that were *collected in safflower fields* (Carthamus tinctorius L.) in Iran. All the listed species were therefore recorded as part of the *herbivore fauna* of that crop.  
No additional host plants, invasive status, disease‑vector role or synonymy are mentioned in the abstract, so the information is limited to what is explicitly stated.  

For each herbivore the following fields were filled:

* **Taxonomy** – taken from standard entomological classification (phylum Arthropoda, class Insecta, order & family as indicated by the family name given in the abstract).  
* **Feeding mode** – the abstract only says the insects are “pest insects” feeding on flower heads, leaves, etc.; therefore the generic mode *herbivory* is used (no specific granivory, frugivory or gall‑forming behaviour is described).  
* **Host plant** – the only plant mentioned in the text is safflower (Carthamus tinctorius L.). All the ins

## Complete pipeline

This was run either for `abstracts_train`, `abstracts_test` or all other abstracts (`scopus_subset`)

In [7]:
# abstracts used for training (prompt fine-tuning)
abstracts_train = np.unique(pd.read_excel("training_set.xlsx")["EID"])
len(abstracts_train), all([a in scopus['EID'].values for a in abstracts_train])

(100, True)

In [8]:
# abstract used for testing 
abstracts_test = np.unique(pd.read_excel("test_set.xlsx")["EID"])
len(abstracts_test), all([a in scopus['EID'].values for a in abstracts_test])

(100, True)

In [9]:
# all other abstracts 
scopus_subset = scopus[~scopus['EID'].isin(abstracts_train) & ~scopus['EID'].isin(abstracts_test)]

In [ ]:
import sys
from datetime import datetime

class Logger(object):
    def __init__(self, filename="log.txt"):
        self.terminal = sys.stdout
        self.log = open(filename, "a", encoding="utf-8")

    def write(self, message):
        if message.strip():
            # Define timestamp each time a message is written
            timestamp = datetime.now().strftime("[%Y-%m-%d %H:%M:%S] ")
            message = "".join([timestamp + line for line in message.splitlines(True)])
        self.terminal.write(message)
        self.log.write(message)
        self.log.flush()

    def flush(self):
        pass  # needed for compatibility with sys.stdout

# Redirect print() output to file:
sys.stdout = Logger("logs/log-complete.txt")

In [ ]:
# Main loop: 

import pandas as pd
from pathlib import Path
import time
import importlib
import functions
importlib.reload(functions)

cols = ["Phylum", "Class", "Order", "Family", "Genus", "Species", "Common Name", "Type", "Feeding Mode",
        "Host Plants", "Herbivore Is Pest", "Pest Importance","Herbivore Is BCA","Herbivore Prey", "Important Natural Enemy","Biological Control",
        "Associated Plants","Associated Industries","Hyperparasitoids", "Invasive","Vectors","Synonyms", "Generated Response", "Abstract", "EID"]

# System + user prompts

# system prompt for generating text: 
with open("prompts/system_prompt_freetext.txt") as f:
    system_prompt_text = f.read()
    
# system prompt for extracting information as JSON: 
with open("prompts/system_prompt_json.txt") as f: 
    system_prompt_json = f.read() 

# user prompt for extracting herbivorous species and their natural enemies: 
with open("prompts/prompt_simple.txt") as f:
    user_prompt_simple = f.read()

# user prompt for extracting information from herbivorous species: 
with open("prompts/prompt_pests.txt") as f:
    user_prompt_herbivores = f.read()

# user prompt for extracting information from natural enemies of herbivorous species: 
with open("prompts/prompt_enemies.txt") as f:
    user_prompt_enemies = f.read()


# -------------------------
# Feather output directory
# -------------------------
feather_dir = Path("feather_results")
feather_dir.mkdir(exist_ok=True)

batch_size = 100
results_buffer = []   
batch_counter = 0

# -------------------------
# Loop setup
# -------------------------
start_time = time.time()
cols = list(cols)  # ensure list type

start_i = 0

for i in range(start_i, len(scopus_subset):

    row = scopus_subset.iloc[i]
    eid = row.get("EID", "")
    title = str(row.get("Title", ""))
    abstract_text = str(row.get("Abstract", ""))
    author_keys = row.get("Author Keywords", "")
    index_keys = row.get("Index Keywords", "")

    abstract = (
        f"{title}. Abstract: {abstract_text}"
        + f". Author keywords: {author_keys}"
        + f". Index keywords: {index_keys}"
    )

    print(f"Iteration {i}  |  EID: {eid}")

    # ============================
    # Extract species list
    # ============================
    species_list, resp_text = functions.get_species_list(
        ollama,
        MODEL, 
        SEED, 
        abstract,
        system_prompt_text,
        system_prompt_json,
        user_prompt_simple,
        prompt_format_species,
    )

    # ============================
    # Build DF for this abstract
    # ============================
    if species_list is None:
        print("species list is empty")
        df = pd.DataFrame([[""] * len(cols)], columns=cols)

    elif (species_list.get("herbivores",[]) == []) and (species_list.get("natural_enemies",[]) == []):
        print("no herbivores or natural enemies contained in list")
        df = pd.DataFrame([[""] * len(cols)], columns=cols)

    else:
        print(
            f"Herbivores: {len(species_list.get('herbivores', []))}"
            f"\tEnemies: {len(species_list.get('natural_enemies', []))}"
        )
        df = functions.get_model_responses(
            ollama,
            MODEL, 
            SEED, 
            system_prompt_text,
            system_prompt_json, 
            species_list, 
            abstract,
            user_prompt_herbivores, 
            user_prompt_enemies,
            prompt_format_herbivore, 
            prompt_format_enemy,
            cols
        )

    # Add metadata
    df["Abstract"] = abstract
    df["EID"] = eid

    # Flatten cells
    df = df.map(functions.flatten_cell)

    # ============================
    # Add to batch buffer
    # ============================
    results_buffer.append(df)
    batch_counter += 1

    # ============================
    # Write batch to Feather
    # ============================
    if batch_counter >= batch_size:
        combined = pd.concat(results_buffer, ignore_index=True)
        outpath = feather_dir / f"batch_{i+1-batch_size}.feather"
        combined.to_feather(outpath)
        print(f"✔ Saved batch: {outpath}")

        # reset buffer
        results_buffer = []
        batch_counter = 0

# -------------------------
# Flush remaining results
# -------------------------
if len(results_buffer) > 0:
    combined = pd.concat(results_buffer, ignore_index=True)
    outpath = feather_dir / f"batch_final.feather"
    combined.to_feather(outpath)
    print(f"✔ Saved final batch: {outpath}")

elapsed = time.time() - start_time
print(f"Finished in {elapsed/60:.2f} minutes")


In [26]:
# Restore normal printing
sys.stdout.log.close()
sys.stdout = sys.stdout.terminal

# Restore the original stdout so print() shows in the notebook again
if hasattr(sys.stdout, "terminal"):
    sys.stdout = sys.stdout.terminal

## Stitch together full dataset

In [ ]:
# add training set and test set to feather results

import pandas as pd
from pathlib import Path
import time
import importlib
import functions
importlib.reload(functions)

feather_dir = Path("feather_results")
feather_dir.mkdir(exist_ok=True)

output_train = pd.read_excel("output-train-gpt-oss-final.xlsx", sheet_name = 1) # these are the corrected rows
output_test = pd.read_excel("output-test-gpt-oss-final.xlsx", sheet_name = 1)   # these are the corrected rows

output_train.to_feather(feather_dir / "output_train.feather")
output_test.to_feather(feather_dir / "output_test.feather")

In [25]:
# create full dataset from all feather results

import pandas as pd
from pathlib import Path
import os 

feather_dir = Path("feather_results")

# delete old dataset if present
file_path = feather_dir / f"dataset_complete.feather"
if os.path.exists(file_path):
    os.remove(file_path)

# Read all feather files and combine
dfs = [
    pd.read_feather(f)
    for f in sorted(feather_dir.glob("*.feather"))
]

full_df = pd.concat(dfs, ignore_index=True)

full_df.to_feather(file_path)

## (Setup for training and test set)

In [ ]:
import sys
from datetime import datetime

class Logger(object):
    def __init__(self, filename="log.txt"):
        self.terminal = sys.stdout
        self.log = open(filename, "a", encoding="utf-8")

    def write(self, message):
        if message.strip():
            # Define timestamp each time a message is written
            timestamp = datetime.now().strftime("[%Y-%m-%d %H:%M:%S] ")
            message = "".join([timestamp + line for line in message.splitlines(True)])
        self.terminal.write(message)
        self.log.write(message)
        self.log.flush()

    def flush(self):
        pass  # needed for compatibility with sys.stdout

# Redirect print() output
sys.stdout = Logger("logs/log-test-set.txt")

In [109]:
# ----------------------------------------------------------------
# Loop through abstracts
start_time = time.time()

i = 0
for eid in abstracts_test:  # abstracts_train
    row = scopus[scopus["EID"] == eid].iloc[0]
    title = str(row.get("Title", ""))
    abstract_text = str(row.get("Abstract", ""))
    author_keys = row.get("Author Keywords", "")
    index_keys = row.get("Index Keywords", "")

    abstract = (
        f"{title}. Abstract: {abstract_text}"
        + (f". Author keywords: {author_keys}") #if pd.notna(author_keys) else "")
        + (f". Index keywords: {index_keys}") #if pd.notna(index_keys) else "")
    )

    print(f"Iteration {i} \tEID: {eid}")

    # get lists of herbivores and natural enemies: 
    species_list, chat_history = get_species_list(abstract, system_prompt, user_prompt_simple, prompt_format_species)
        
    if species_list is None: 
            print("species list is empty")
    else:
        if (species_list.get("herbivores",[]) == []) & (species_list.get("natural_enemies",[]) == []): 
            df = pd.DataFrame([[""] * len(cols)],columns=cols)
            print("no herbivores or natural enemies contained in list")
            print(species_list)
        else: 
            print(f"Herbivores: {len(species_list.get("herbivores",[]))} \t\t Enemies: {len(species_list.get("natural_enemies",[]))}")
            df = get_model_responses(system_prompt, species_list, abstract, prompt_format_herbivore, prompt_format_enemy, cols)
            
    df["Abstract"] = abstract
    df["EID"] = eid

    df = df.map(flatten_cell)

    for j, row in enumerate(df.itertuples(index=False), start=1):
        ws.append(row)

    wb.save(output_file) 
    i+=1


elapsed = time.time() - start_time
print(f"Finished in {elapsed/60:.2f} minutes")

In [ ]:
# Restore normal printing
sys.stdout.log.close()
sys.stdout = sys.stdout.terminal

# Restore the original stdout so print() shows in the notebook again
if hasattr(sys.stdout, "terminal"):
    sys.stdout = sys.stdout.terminal

In [1599]:
print("gpt-oss:120b")
print("time taken for 100 abstracts: 46.25 minutes") # 114.81
print("total estimated time for ~100,000 abstracts: ", round(46.25*1128.31/60/24,1), "days.")

gpt-oss:120b
time taken for 100 abstracts: 46.25 minutes
total estimated time for ~100,000 abstracts:  36.2 days.
